# Pengumpulan Data

In [ ]:
#@title Twitter Auth Token

twitter_auth_token = 'ccb0a6dc47226f7ddc27a7a0975107094c8899e3'

In [ ]:
# Install Node.js
%sudo apt-get update
%sudo apt-get install -y ca-certificates curl gnupg
%sudo mkdir -p /etc/apt/keyrings
%curl -fsSL https://deb.nodesource.com/gpgkey/nodesource-repo.gpg.key | sudo gpg --dearmor -o /etc/apt/keyrings/nodesource.gpg

%NODE_MAJOR=20 && echo "deb [signed-by=/etc/apt/keyrings/nodesource.gpg] https://deb.nodesource.com/node_$NODE_MAJOR.x nodistro main" | sudo tee /etc/apt/sources.list.d/nodesource.list

%sudo apt-get update
%sudo apt-get install nodejs -y

%node -v

In [ ]:
# Crawl Data
import os

# List keyword untuk digunakan
keywords = [
    'gaji gen z lang:id until:2024-12-31 since:2020-01-01',
    'kesehatan mental generasi z lang:id until:2024-12-31 since:2020-01-01'
    'finansial gen z lang:id until:2024-12-31 since:2020-01-01'
]

# Filename untuk setiap keyword
filenames = [
    'dataset-gaji-gen-z.csv',
    'dataset_kesehatan_mental_generasi_z.csv',
    'dataset_finansial_gen_z.csv'
]

# Limit data yang ingin dikumpulkan
limit = 50000

# Lakukan crawling untuk setiap keyword
for keyword, filename in zip(keywords, filenames):
    os.system(f'npx -y tweet-harvest@2.6.1 -o "{filename}" -s "{keyword}" --tab "LATEST" -l {limit} --token {twitter_auth_token}')

# Pra-pemrosesan data

In [2]:
import pandas as pd

# Membaca dataset
df1 = pd.read_csv('REAL-DATA/REAL-dataset_finansial_gen_z.csv')
df2 = pd.read_csv('REAL-DATA/REAL-dataset_gaji_gen_z.csv')
df3 = pd.read_csv('REAL-DATA/REAL-dataset_kesehatan_mental_generasi_z.csv')
df4 = pd.read_csv('data matang/dataset_finansial_gen_z.csv')
df5 = pd.read_csv('data matang/dataset_gaji_gen_z.csv')
df6 = pd.read_csv('data matang/dataset_kesehatan_mental_generasi_z-TGL.csv')

# Menambahkan kolom keyword
df1['keyword'] = 'finansial gen z'
df2['keyword'] = 'gaji gen z'
df3['keyword'] = 'kesehatan mental generasi z'
df4['keyword'] = 'finansial gen z'
df5['keyword'] = 'gaji gen z'
df6['keyword'] = 'kesehatan mental generasi z'

# Menggabungkan data set berdasarkan keyword
df_finansial = pd.concat([df1, df4], ignore_index=True)
df_gaji = pd.concat([df2, df5], ignore_index=True)
df_kesehatan_mental = pd.concat([df3, df6], ignore_index=True)

# Menggabungkan semua dataframe menjadi satu
merge_df = pd.concat([df_finansial, df_gaji, df_kesehatan_mental], ignore_index=True)

# Menyimpan ke CSV
merge_df.to_csv('data-analisis/datasets-kotor.csv', index=False)

In [3]:
df = pd.read_csv('data-analisis/datasets-kotor.csv')
df.count()

In [4]:
df = pd.read_csv('data-analisis/datasets-kotor.csv')
df.count()

In [5]:
#drop column is not needed
df.drop(df.columns[[0,2,4,5,6,7,8,9,10,11,12,11,13,14]], axis=1, inplace=True)

df.to_csv('data-analisis/datasets.csv', index=False)

In [6]:
df.count()

In [7]:
# drop row have duplicate value
df.drop_duplicates(subset=['full_text'], inplace=True)

#drop row have missing value
df.dropna(subset=['full_text'], inplace=True)

df.to_csv('data-analisis/datasets.csv', index=False)
df.count()

## cleansing data

In [8]:
import pandas as pd
df = pd.read_csv('data-analisis/datasets.csv')

In [9]:
df

In [10]:
# cleansing data
import re
import string

def clean_text(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # delete mention
    text = re.sub(r'#', '', text) # delete hastag
    text = re.sub(r'RT[\s]+', '', text) # delate RT
    text = re.sub(r'https?:\/\/\S+', '', text) # delete hyperlink
    text = re.sub(r'\n', '', text) # delete new line
    text = re.sub(r'\d+', '', text) # delete number
    text = re.sub(r'[^A-Za-z ]+', '', text) # delete non alphabet
    
    text = text.replace('…', '') # delete ellipsis
    text = text.translate(str.maketrans('', '', string.punctuation)) # delete punctuation
    text = text.strip() # delete space
    return text

df ['cleanning_text'] = df['full_text'].apply(clean_text)
df.head()

## case folding

In [11]:
# case folding
def case_folding(text):
    text = text.lower() # change to lower case
    return text

df['case_folding'] = df['cleanning_text'].apply(case_folding)
df

## convert slang word

In [12]:
# convert slang word
slang_words = pd.read_csv('https://raw.githubusercontent.com/nasalsabila/kamus-alay/refs/heads/master/colloquial-indonesian-lexicon.csv')
slang_words_dict = dict(zip(slang_words['slang'], slang_words['formal']))

# Fungsi untuk mengonversi slang word
def convert_slang_word(text):
    return ' '.join([slang_words_dict.get(word, word) for word in text.split()])

# Menerapkan fungsi ke kolom 'case_folding'
df['convert_slang_word'] = df['case_folding'].apply(convert_slang_word)
df

## Stop word

In [13]:
# Stop word
# from nltk.corpus import stopwords

# def filtering(text):
#     stop_words = set(stopwords.words('indonesian'))
#     word_tokens = text.split()
#     text = [word for word in word_tokens if word not in stop_words]
#     text = ' '.join(text)
#     return text

# df['filtering'] = df['convert_slang_word'].apply(filtering)
# df

# Stop word
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

def filtering(text):
    factory = StopWordRemoverFactory()
    stop_words = set(factory.get_stop_words())
    word_tokens = text.split()
    text = [word for word in word_tokens if word not in stop_words]
    text = ' '.join(text)
    return text

df['filtering'] = df['convert_slang_word'].apply(filtering)
df

## tokenizing

In [14]:
import nltk
# nltk.download('punkt')

In [15]:
# tokenizing
from nltk.tokenize import word_tokenize

def tokenizing(text):
    text = word_tokenize(text)
    return text

df['tokenizing'] = df['filtering'].apply(tokenizing)
df

## stemming

In [24]:
# stemming
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# create stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stem_text(tokens):
    text = ' '.join(tokens)  # Join the tokens back into a single string
    return stemmer.stem(text)

df['stemming'] = df['tokenizing'].apply(stem_text)
df

In [25]:
#save to csv
df.to_csv('data-analisis/datasets.csv', index=False)

In [26]:
df.info()

In [27]:
df.drop(df.columns[[1, 2, 3, 4, 5, 6]], axis=1, inplace=True)

# Rename column 'stemming' to 'full_text'
df.rename(columns={'stemming': 'full_text'}, inplace=True)

df.to_csv('data-analisis/datasets-clean.csv', index=False)

# Pelabelan Data

In [28]:
import pandas as pd
df = pd.read_csv('data-analisis/datasets-clean.csv')
df.head()

In [29]:
df.info()

In [30]:
import pandas as pd

#unduh kamus inset lexicon positif dan negatif
positive_url = "https://raw.githubusercontent.com/fajri91/InSet/master/positive.tsv"
negative_url = "https://raw.githubusercontent.com/fajri91/InSet/master/negative.tsv"

positive_lexicon = set(pd.read_csv(positive_url, sep='\t', header=None)[0])
negative_lexicon = set(pd.read_csv(negative_url, sep='\t', header=None)[0])

#fungsi menghitung skor sentimen
def determine_sentiment(text):
    if isinstance(text, str):
        positive_count = sum(1 for word in text.split() if word in positive_lexicon)
        negative_count = sum(1 for word in text.split() if word in negative_lexicon)
        sentiment_score = positive_count - negative_count
        if sentiment_score > 0:
            sentiment = 'Positif'
        elif sentiment_score < 0:
            sentiment = 'Negatif'
        else:
            sentiment = 'Netral'
        return sentiment_score, sentiment
    return 0, "netral"
        
#menerapkan perhitungan ke datasets
df[['score', 'label']] = df['full_text'].apply(lambda x: pd.Series(determine_sentiment(x)))

df.to_csv('data-analisis/datasets-label.csv', index=False)


In [ ]:
# Menghitung jumlah label
df['label'].value_counts()

# Ekstraksi Fitur

In [32]:
import pandas as pd
#Membaca dataset yang sudah diberi label
df = pd.read_csv('data-analisis/datasets-label.csv')
df['label'].value_counts()

versi yutub

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

#menggunakan TFidVectorizer untuk menghitung TF - IDF
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['full_text'])

#menghitung IDF
term = tfidf_vectorizer.get_feature_names_out()
idf = np.log(tfidf_matrix.shape[0] / (np.count_nonzero(tfidf_matrix.toarray(), axis=0) + 1))

#membuat data frame untuk menyimpan term dan idf
tfidf_df = pd.DataFrame({'term' : term, 'idf' : idf})

#tambah ke kolom TF data frame
for i, doc in enumerate(df['full_text']):
    tf = tfidf_matrix[i].toarray().flatten()
    tfidf_df[f'tf_{i}'] = tf
    
tfidf_df.to_csv('REAL-DATA/datasets-tfidfshow.csv', index=False)

versi riset

In [33]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv('data-analisis/datasets-label.csv')
# Convert text to vectors using TF-IDF
tfidf_vectorizer = TfidfVectorizer()
x = tfidf_vectorizer.fit_transform(df['full_text'])

tfidf = x.toarray()
print(tfidf[:1])

# Convert the array to a DataFrame
tfidf_df = pd.DataFrame(tfidf)

# Save the DataFrame to a CSV file
tfidf_df.to_csv('data-analisis/datasets-tfidf.csv', index=False)

# save to pickle
# tfidf_df.to_pickle('REAL-DATA/datasets-tfidf.pkl')

# Data Balancing

In [34]:
df.isna().sum() # to check null values

In [35]:
import pandas as pd

tfidf_data = pd.read_csv("data-analisis/datasets-tfidf.csv")
labels_data = pd.read_csv("data-analisis/datasets-label.csv")

# # Check for null values in both datasets
# print("Null values in tfidf_data:")
# print(tfidf_data.isna().sum())
# print("\nNull values in labels_data:")
# print(labels_data.isna().sum())

# Drop rows with null values in labels_data
labels_data = labels_data.dropna(subset=['label'])

# Merge the TF-IDF features with the labels
data = pd.concat([tfidf_data, labels_data['label']], axis=1)
data.to_csv('data-analisis/datasets-balance.csv', index=False)
print(data.head())

In [36]:
df = pd.read_csv('data-analisis/datasets-balance.csv')
df['label'].value_counts()

In [37]:
# Import library yang diperlukan
import pandas as pd
from imblearn.over_sampling import SMOTE

# Membaca dataset yang sudah digabungkan
df_combined = pd.read_csv('data-analisis/datasets-balance.csv')

# Memisahkan fitur dan label
X = df_combined.drop(columns=['label'])  # Menghapus kolom label
y = df_combined['label']  # Mengambil kolom label

# Menggunakan SMOTE untuk melakukan data balancing
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Menampilkan jumlah kelas setelah balancing
print("Jumlah kelas sebelum SMOTE:")
print(y.value_counts())
print("\nJumlah kelas setelah SMOTE:")
print(y_resampled.value_counts())

# Menyimpan dataset yang sudah di-balance ke file CSV
balanced_df = pd.concat([pd.DataFrame(X_resampled, columns=X.columns), pd.DataFrame(y_resampled, columns=['label'])], axis=1)
balanced_df.to_csv('data-analisis/datasets-balanced.csv', index=False)

# Untuk tahapan modeling, dilakukang pada notebook berbeda

# eksport Data Untuk Dashboard

In [ ]:
import pandas as pd
from nltk.tokenize import word_tokenize
from collections import Counter

df=pd.read_csv('REAL-DATA/datasets-clean.csv')

def create_word_count_table(df, text_column):
    # Tokenizing the text data
    df['tokens'] = df[text_column].apply(word_tokenize)
    
    # Flatten the list of tokens and count the occurrences of each word
    all_tokens = [token for sublist in df['tokens'] for token in sublist]
    word_counts = Counter(all_tokens)
    
    # Convert the word counts to a DataFrame
    word_count_df = pd.DataFrame(word_counts.items(), columns=['word', 'count'])
    
    return word_count_df

word_tokenize_df = create_word_count_table(df, 'full_text')
word_tokenize_df.to_csv('word_count_result.csv', index=False)

In [ ]:
df = pd.read_csv('word_count_result.csv')

df.isnull().sum()

# Mencari baris yang memiliki nilai NaN atau kosong
df[df.isnull().any(axis=1)]

# Menampilkan baris yang memiliki nilai NaN atau kosong
# print(rows_with_nan)

df.dropna(inplace=True)

In [ ]:
df = pd.read_csv('word_count_result.csv')

positive_lexicon = pd.read_csv('InSet/positive.tsv', sep='\t', header=None)
negative_lexicon = pd.read_csv('InSet/negative.tsv', sep='\t', header=None)

# Gabungkan lexicon positif dan negatif
positive_lexicon.columns = ['kata', 'polaritas']
negative_lexicon.columns = ['kata', 'polaritas']

# Pastikan kolom polaritas bertipe numerik
positive_lexicon['polaritas'] = pd.to_numeric(positive_lexicon['polaritas'], errors='coerce')
negative_lexicon['polaritas'] = pd.to_numeric(negative_lexicon['polaritas'], errors='coerce')

lexicon = pd.concat([positive_lexicon, negative_lexicon])

# Konversi lexicon ke dictionary untuk lookup cepat
lexicon_dict = dict(zip(lexicon['kata'], lexicon['polaritas']))

# Fungsi untuk memberikan skor pada teks berdasarkan kamus lexicon
def label(tweet, lexicon_dict):
    words = tweet.split()  # Pisahkan tweet menjadi kata-kata
    sentiment_score = 0  # Inisialisasi skor sentimen

    # Hitung skor sentimen berdasarkan kata-kata dalam lexicon
    for word in words:
        sentiment = lexicon_dict.get(word, 0)  # Ambil polaritas dari dictionary, default 0 jika tidak ditemukan
        sentiment_score += sentiment

    # Berikan label berdasarkan skor polaritas total
    if sentiment_score > 0:
        return 'positif', sentiment_score
    elif sentiment_score < 0:
        return 'negatif', sentiment_score
    else:
        return 'netral', sentiment_score

# Handle NaN values in the 'word' column
df['word'].fillna('', inplace=True)

df['label', 'score'] = df['word'].apply(lambda x: pd.Series(label(x, lexicon_dict)))
df.to_csv('word_count_labeled.csv', index=False)

In [ ]:
import pandas as pd

df = pd.read_csv('word_count_labeled.csv')

df['label'].value_counts()

df.groupby('label').apply(lambda x: x.loc[x['count'].idxmax()])

In [ ]:
# import pandas as pd
# from wordcloud import WordCloud, get_single_color_func
# import matplotlib.pyplot as plt

# df = pd.read_csv('word_count_labeled.csv')

# # Menampilkan jumlah label
# # print(df['label'].value_counts())

# # Fungsi untuk membuat dan menampilkan Word Cloud dengan warna berdasarkan label
# def plot_word_cloud(label, color):
#     words = df[df['label'] == label].set_index('word')['count'].to_dict()
#     wordcloud = WordCloud(width=800, height=400, background_color='white', color_func=get_single_color_func(color)).generate_from_frequencies(words)
    
#     plt.figure(figsize=(10, 5))
#     plt.imshow(wordcloud, interpolation='bilinear')
#     plt.title(f'Word Cloud for {label} words')
#     plt.axis('off')
#     plt.show()

# # Menampilkan Word Cloud untuk setiap label dengan warna yang sesuai
# label_colors = {
#     'positif': 'green',
#     'negatif': 'red',
#     'netral': 'gray'
# }

# for label in df['label'].unique():
#     plot_word_cloud(label, label_colors[label])

In [1]:
import pandas as pd

eval_svm = pd.read_csv('HASIL-RISET/evaluation_results_SVM-new.csv')
eval_nb = pd.read_csv('HASIL-RISET/evaluation_results_nb-new.csv')
eval_knn = pd.read_csv('HASIL-RISET/evaluation_results_knn-new.csv')

# penggabungan data evaluation
# Menambahkan kolom 'model' ke setiap DataFrame
eval_svm['model'] = 'svm'
eval_nb['model'] = 'nb'
eval_knn['model'] = 'knn'

# Mengatur ulang kolom agar 'model' menjadi kolom pertama
svm = eval_svm[['model'] + [col for col in eval_svm.columns if col != 'model']]
nb = eval_nb[['model'] + [col for col in eval_nb.columns if col != 'model']]
knn = eval_knn[['model'] + [col for col in eval_knn.columns if col != 'model']]

# Menggabungkan semua DataFrame
combined_df = pd.concat([svm, nb, knn], axis=0, ignore_index=True)

# Menampilkan hasil
print(combined_df.head())

combined_df.to_csv('HASIL-RISET/evaluation_results_combine.csv', index=False)